# Testing Knowledge Distillation
Questo notebook serve esclusivamente a testare e valutare i modelli salvati in precedenza, confrontando la Baseline e lo Student Distillato. Non viene effettuato alcun addestramento.

In [ ]:
!pip install -q transformers datasets evaluate rouge_score bert_score matplotlib accelerate

## 1. Configurazione

In [ ]:
import os

TASK_TYPE = "summarization" # o "qa"
STUDENT_BASELINE_ID = "HuggingFaceTB/SmolLM-135M"
STUDENT_DISTILLED_PATH = f"./student_distilled_{TASK_TYPE}_final"
TEACHER_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # Solo per avere la formattazione corretta del teacher se servisse

MAX_TEST_SAMPLES = "all" # Usa "all" o un intero (es. 100)
EVAL_BATCH_SIZE = 8


## 2. Preparazione dei Dati (Test Set)

In [ ]:
from datasets import load_dataset

print(f"Caricamento test set per il task: {TASK_TYPE}")

if TASK_TYPE == "summarization":
    dataset = load_dataset("knkarthick/samsum")
    def format_teacher_prompt(example):
        messages = [
            {"role": "system", "content": "You are a highly accurate summarization assistant. Provide a concise summary of the following conversation."},
            {"role": "user", "content": f"Summarize this dialogue:\n\n{example['dialogue']}"}
        ]
        return {"teacher_prompt_messages": messages, "target": example['summary'], "input_text": example['dialogue']}
elif TASK_TYPE == "qa":
    dataset = load_dataset("databricks/databricks-dolly-15k")
    # Dolly ha solo la partizione 'train', facciamo uno split 90/10 manuale fissando il seed
    dataset = dataset['train'].train_test_split(test_size=0.1, seed=42)
    
    def format_teacher_prompt(example):
        instruction = example['instruction']
        context = example['context']
        if context and context.strip():
            user_content = f"Context: {context}\n\nInstruction: {instruction}"
            input_text = f"Context: {context[:300]}...\nInstruction: {instruction}"
        else:
            user_content = f"Instruction: {instruction}"
            input_text = f"Instruction: {instruction}"
            
        messages = [
            {"role": "system", "content": "You are a helpful and concise AI assistant. Follow the user's instruction."},
            {"role": "user", "content": user_content}
        ]
        return {"teacher_prompt_messages": messages, "target": example['response'], "input_text": input_text}


## 3. Valutazione e Profilazione (Batched)

In [ ]:
import evaluate
import time
import math
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

rouge_metric = evaluate.load("rouge")
bert_metric = evaluate.load("bertscore")

# Setup template studente (usato come base)
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_BASELINE_ID)
if student_tokenizer.chat_template is None:
    student_tokenizer.chat_template = (
        "{% for message in messages %}"
        "<|im_start|>{{ message['role'] }}\n"
        "{{ message['content'] }}<|im_end|>\n"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "<|im_start|>assistant\n"
        "{% endif %}"
    )

def profile_and_evaluate(model_id, is_baseline=False):
    print(f"\nValutazione modello: {model_id}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
    except:
        print("Tokenizer non trovato nella cartella, uso quello della baseline.")
        tokenizer = AutoTokenizer.from_pretrained(STUDENT_BASELINE_ID)
        
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left' # Per generazione in batch
    
    if tokenizer.chat_template is None:
        tokenizer.chat_template = student_tokenizer.chat_template

    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, device_map="auto")
    except Exception as e:
        print(f"Errore caricamento modello {model_id}: {e}")
        return None
        
    model.eval()
    
    predictions = []
    references = []
    total_time = 0
    total_generated_tokens = 0
    total_loss = 0.0
    
    for i in tqdm(range(0, len(test_data), EVAL_BATCH_SIZE), desc="Evaluating"):
        batch = test_data[i:i+EVAL_BATCH_SIZE]
        
        prompt_strs = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch["teacher_prompt_messages"]]
        
        # Perplexity calculation: richiede i full target uniti
        full_texts = [p + t + tokenizer.eos_token for p, t in zip(prompt_strs, batch['target'])]
        full_inputs = tokenizer(full_texts, return_tensors="pt", padding=True).to(model.device)
        
        # Loss forward
        with torch.no_grad():
            loss_output = model(**full_inputs, labels=full_inputs["input_ids"])
            total_loss += loss_output.loss.item() * len(batch['target']) # Approssimiamo moltiplicando per bs
            
            inputs = tokenizer(prompt_strs, return_tensors="pt", padding=True).to(model.device)
            
            start_time = time.time()
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
            end_time = time.time()
            
        generated_tokens = outputs[:, inputs['input_ids'].shape[1]:]
        gen_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        
        total_time += (end_time - start_time)
        total_generated_tokens += torch.sum(generated_tokens != tokenizer.pad_token_id).item()
        
        predictions.extend([g.strip() for g in gen_texts])
        references.extend(batch['target'])
        
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    bert_results = bert_metric.compute(predictions=predictions, references=references, lang="en")
    mean_bert_f1 = sum(bert_results['f1']) / len(bert_results['f1'])
    
    avg_loss = total_loss / len(test_data)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float('inf')
    ms_per_token = (total_time * 1000) / total_generated_tokens if total_generated_tokens > 0 else 0
    param_count = sum(p.numel() for p in model.parameters()) / 1e6
    
    del model
    torch.cuda.empty_cache()
    
    return {
        "Perplexity": perplexity,
        "ROUGE-L": rouge_results['rougeL'],
        "BERTScore-F1": mean_bert_f1,
        "Latency/Token (ms)": ms_per_token,
        "Parameters (M)": param_count,
        "predictions": predictions
    }

res_baseline = profile_and_evaluate(STUDENT_BASELINE_ID, is_baseline=True)
res_distilled = profile_and_evaluate(STUDENT_DISTILLED_PATH)

if res_distilled is not None:
    print("\n=== RISULTATI COMPARATIVI ===")
    print("Student Baseline (Zero-Shot):")
    for k, v in res_baseline.items():
        if k != "predictions": print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
            
    print("\nStudent Distilled:")
    for k, v in res_distilled.items():
        if k != "predictions": print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


## 4. Grafici per Relazione

In [ ]:
if res_distilled is not None:
    import matplotlib.pyplot as plt
    
    metrics = ['Perplexity', 'ROUGE-L', 'BERTScore-F1', 'Latency/Token (ms)']
    baseline_vals = [res_baseline[m] for m in metrics]
    distilled_vals = [res_distilled[m] for m in metrics]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, baseline_vals, width, label='Student Baseline ZS', color='lightcoral')
    rects2 = ax.bar(x + width/2, distilled_vals, width, label='Student Distilled', color='mediumseagreen')
    
    ax.set_ylabel('Scores / Valori')
    ax.set_title('Confronto Prestazioni: Baseline vs Distilled')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.2f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')
    
    autolabel(rects1)
    autolabel(rects2)
    
    fig.tight_layout()
    plt.show()


## 5. Ispezione Qualitativa (Esempi Finali)

In [ ]:
if res_distilled is not None:
    print("=== ISPEZIONE QUALITATIVA DEGLI OUTPUT ===\n")
    import random
    
    indices = random.sample(range(len(test_data)), min(3, len(test_data)))
    
    for i, idx in enumerate(indices, 1):
        sample = test_data[idx]
        pred_baseline = res_baseline['predictions'][idx]
        pred_distilled = res_distilled['predictions'][idx]
        
        print(f"--- SAMPLE {i} ---")
        print(f"INPUT:\n{sample['input_text'].strip()}\n")
        print(f"TARGET IDEALE (Human):\n{sample['target'].strip()}\n")
        print(f"STUDENT BASELINE (Zero-Shot):\n{pred_baseline.strip()}\n")
        print(f"STUDENT DISTILLED:\n{pred_distilled.strip()}\n")
        print("="*80 + "\n")
